In [1]:
import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42

# Target
target = "satisfaction"

# Features
drop_cols = [target, "id"]

X = train.drop(columns=drop_cols).copy()
y = train[target].astype(int)

X_test = test.drop(columns=["id"]).copy()

# Categorical columns
cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

# Same validation split for every model
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test :", X_test.shape)
print("Positive rate:", y.mean())

Train: (559708, 21)
Valid: (139927, 21)
Test : (299844, 21)
Positive rate: 0.44357272006117476


In [3]:
# ============================================================
# PHASE 3A — LOAD ALL SAVED PREDICTIONS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------
# Load validation predictions
# ------------------------------------------------------------

cat_valid = np.load("artifacts/oof/baseline_cat.npy")
lgb_valid = np.load("artifacts/oof/baseline_lgb.npy")
xgb_valid = np.load("artifacts/oof/baseline_xgb.npy")
lgb_opt_valid = np.load("artifacts/oof/lgb_optimized.npy")

# ------------------------------------------------------------
# Load test predictions
# ------------------------------------------------------------

cat_test = np.load("artifacts/test/baseline_cat.npy")
lgb_test = np.load("artifacts/test/baseline_lgb.npy")
xgb_test = np.load("artifacts/test/baseline_xgb.npy")
lgb_opt_test = np.load("artifacts/test/lgb_optimized.npy")

# ------------------------------------------------------------
# Verify shapes
# ------------------------------------------------------------

print("VALIDATION SHAPES")
print("CatBoost :", cat_valid.shape)
print("LightGBM :", lgb_valid.shape)
print("XGBoost  :", xgb_valid.shape)
print("LGB Opt  :", lgb_opt_valid.shape)

print("\nTEST SHAPES")
print("CatBoost :", cat_test.shape)
print("LightGBM :", lgb_test.shape)
print("XGBoost  :", xgb_test.shape)
print("LGB Opt  :", lgb_opt_test.shape)

assert len(cat_valid) == len(y_valid)
assert len(lgb_valid) == len(y_valid)
assert len(xgb_valid) == len(y_valid)
assert len(lgb_opt_valid) == len(y_valid)

assert len(cat_test) == len(X_test)
assert len(lgb_test) == len(X_test)
assert len(xgb_test) == len(X_test)
assert len(lgb_opt_test) == len(X_test)

print("\n✓ All prediction arrays align correctly.")

VALIDATION SHAPES
CatBoost : (139927,)
LightGBM : (139927,)
XGBoost  : (139927,)
LGB Opt  : (139927,)

TEST SHAPES
CatBoost : (299844,)
LightGBM : (299844,)
XGBoost  : (299844,)
LGB Opt  : (299844,)

✓ All prediction arrays align correctly.


## Experiment - 1A => Service Aggregates

In [4]:
# ============================================================
# PHASE 5 — EXP 1
# SERVICE-RATING FEATURE DISCOVERY
# ============================================================

print("All columns:")
for i, col in enumerate(train.columns):
    print(f"{i:2d}. {col}")

# Automatically identify likely service/rating columns
rating_candidates = []

for col in train.columns:
    name = col.lower()

    if any(keyword in name for keyword in [
        "service",
        "cleanliness",
        "comfort",
        "food",
        "entertainment",
        "boarding",
        "seat",
        "wifi",
        "inflight",
        "checkin",
        "online",
        "baggage",
        "legroom"
    ]):
        rating_candidates.append(col)

print("\nPotential service/rating columns:")
for col in rating_candidates:
    print(col)

print(f"\nCount: {len(rating_candidates)}")

All columns:
 0. id
 1. Age
 2. Flight Distance
 3. Inflight wifi service
 4. Departure/Arrival time convenient
 5. Ease of Online booking
 6. Gate location
 7. Food and drink
 8. Online boarding
 9. Seat comfort
10. Inflight entertainment
11. On-board service
12. Leg room service
13. Baggage handling
14. Checkin service
15. Cleanliness
16. Departure Delay in Minutes
17. Arrival Delay in Minutes
18. Gender
19. Customer Type
20. Type of Travel
21. Class
22. satisfaction

Potential service/rating columns:
Inflight wifi service
Ease of Online booking
Food and drink
Online boarding
Seat comfort
Inflight entertainment
On-board service
Leg room service
Baggage handling
Checkin service
Cleanliness

Count: 11


In [5]:
# ============================================================
# PHASE 5 — EXP 1A
# SERVICE-RATING AGGREGATES
# ============================================================

service_cols = [
    "Inflight wifi service",
    "Ease of Online booking",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness"
]

def add_service_features(df):
    df = df.copy()

    service_data = df[service_cols]

    df["service_sum"] = service_data.sum(axis=1)
    df["service_mean"] = service_data.mean(axis=1)
    df["service_std"] = service_data.std(axis=1)
    df["service_min"] = service_data.min(axis=1)
    df["service_max"] = service_data.max(axis=1)
    df["service_median"] = service_data.median(axis=1)

    df["poor_service_count"] = (service_data <= 2).sum(axis=1)
    df["excellent_service_count"] = (service_data >= 4).sum(axis=1)

    return df


X_train_fe = add_service_features(X_train)
X_valid_fe = add_service_features(X_valid)
X_test_fe  = add_service_features(X_test)

print("Original features:", X_train.shape[1])
print("New features:", X_train_fe.shape[1])

new_features = [
    "service_sum",
    "service_mean",
    "service_std",
    "service_min",
    "service_max",
    "service_median",
    "poor_service_count",
    "excellent_service_count"
]

print("\nCreated:")
print(new_features)

print("\nSample:")
print(X_train_fe[new_features].head())

Original features: 21
New features: 29

Created:
['service_sum', 'service_mean', 'service_std', 'service_min', 'service_max', 'service_median', 'poor_service_count', 'excellent_service_count']

Sample:
        service_sum  service_mean  service_std  service_min  service_max  \
159366           45      4.090909     1.578261            1            5   
659958           21      1.909091     1.221028            1            4   
434666           29      2.636364     1.286291            1            4   
427957           51      4.636364     0.674200            3            5   
409049           39      3.545455     1.213560            2            5   

        service_median  poor_service_count  excellent_service_count  
159366             5.0                   2                        9  
659958             1.0                   8                        2  
434666             3.0                   5                        4  
427957             5.0                   0                   

In [6]:
cat_cols_fe = [
    col for col in cat_cols
    if col in X_train_fe.columns
]

for col in cat_cols_fe:
    X_train_fe[col] = X_train_fe[col].astype("category")
    X_valid_fe[col] = X_valid_fe[col].astype("category")
    X_test_fe[col]  = X_test_fe[col].astype("category")

In [9]:
# ============================================================
# EXP 1A — LIGHTGBM WITH SERVICE AGGREGATES
# ============================================================

import lightgbm as lgb
from sklearn.metrics import roc_auc_score

lgb_fe = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,

    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=0.5,

    random_state=SEED,
    n_jobs=-1,

    verbosity=-1
)

lgb_fe.fit(
    X_train_fe,
    y_train,
    categorical_feature=cat_cols_fe,

    eval_set=[(X_valid_fe, y_valid)],
    callbacks=[
        lgb.early_stopping(100, verbose=False)
    ]
)

fe_valid_pred = lgb_fe.predict_proba(
    X_valid_fe
)[:, 1]

fe_auc = roc_auc_score(
    y_valid,
    fe_valid_pred
)

baseline_auc = 0.958134780

print("=" * 60)
print("EXP 1A — SERVICE AGGREGATES")
print("=" * 60)

print(f"Baseline LGB AUC : {baseline_auc:.9f}")
print(f"Feature LGB AUC  : {fe_auc:.9f}")
print(f"Delta             : {fe_auc - baseline_auc:+.9f}")
print(f"Best iteration    : {lgb_fe.best_iteration_}")

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


EXP 1A — SERVICE AGGREGATES
Baseline LGB AUC : 0.958134780
Feature LGB AUC  : 0.957823434
Delta             : -0.000311346
Best iteration    : 497


## EXP 1B — Service Domain Features

In [10]:
# ============================================================
# PHASE 5 — EXP 1B
# SERVICE DOMAIN FEATURES
# ============================================================

digital_cols = [
    "Inflight wifi service",
    "Ease of Online booking",
    "Online boarding"
]

cabin_cols = [
    "Food and drink",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Cleanliness"
]

airport_cols = [
    "Baggage handling",
    "Checkin service"
]


def add_domain_features(df):
    df = df.copy()

    df["digital_experience"] = df[digital_cols].mean(axis=1)
    df["cabin_experience"] = df[cabin_cols].mean(axis=1)
    df["airport_handling"] = df[airport_cols].mean(axis=1)

    return df


X_train_fe = add_domain_features(X_train)
X_valid_fe = add_domain_features(X_valid)
X_test_fe  = add_domain_features(X_test)

domain_features = [
    "digital_experience",
    "cabin_experience",
    "airport_handling"
]

print("Created features:")
print(domain_features)

print("\nSample:")
print(X_train_fe[domain_features].head())

Created features:
['digital_experience', 'cabin_experience', 'airport_handling']

Sample:
        digital_experience  cabin_experience  airport_handling
159366            2.333333          4.666667               5.0
659958            1.666667          1.500000               3.5
434666            1.666667          3.333333               2.0
427957            4.666667          4.666667               4.5
409049            2.000000          4.000000               4.5


In [11]:
cat_cols_fe = [
    col for col in cat_cols
    if col in X_train_fe.columns
]

for col in cat_cols_fe:
    X_train_fe[col] = X_train_fe[col].astype("category")
    X_valid_fe[col] = X_valid_fe[col].astype("category")
    X_test_fe[col]  = X_test_fe[col].astype("category")

In [12]:
# ============================================================
# EXP 1B — LIGHTGBM WITH DOMAIN FEATURES
# ============================================================

lgb_domain = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,

    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=0.5,

    random_state=SEED,
    n_jobs=-1
)

lgb_domain.fit(
    X_train_fe,
    y_train,
    categorical_feature=cat_cols_fe,

    eval_set=[(X_valid_fe, y_valid)],

    callbacks=[
        lgb.early_stopping(100, verbose=False)
    ]
)

domain_valid_pred = lgb_domain.predict_proba(
    X_valid_fe
)[:, 1]

domain_auc = roc_auc_score(
    y_valid,
    domain_valid_pred
)

print("=" * 60)
print("EXP 1B — SERVICE DOMAIN FEATURES")
print("=" * 60)

print(f"Baseline LGB AUC : {0.958134780:.9f}")
print(f"Domain LGB AUC   : {domain_auc:.9f}")
print(f"Delta             : {domain_auc - 0.958134780:+.9f}")
print(f"Best iteration    : {lgb_domain.best_iteration_}")

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


EXP 1B — SERVICE DOMAIN FEATURES
Baseline LGB AUC : 0.958134780
Domain LGB AUC   : 0.958074046
Delta             : -0.000060734
Best iteration    : 670


## EXP 2 — Delay Engineering

In [13]:
# ============================================================
# PHASE 5 — EXP 2
# DELAY FEATURES
# ============================================================

import numpy as np

def add_delay_features(df):
    df = df.copy()

    dep = df["Departure Delay in Minutes"].fillna(0)
    arr = df["Arrival Delay in Minutes"].fillna(0)

    df["total_delay"] = dep + arr
    df["delay_difference"] = arr - dep

    df["has_departure_delay"] = (dep > 0).astype("int8")
    df["has_arrival_delay"] = (arr > 0).astype("int8")

    df["log_total_delay"] = np.log1p(df["total_delay"])

    return df


X_train_delay = add_delay_features(X_train)
X_valid_delay = add_delay_features(X_valid)
X_test_delay  = add_delay_features(X_test)

delay_features = [
    "total_delay",
    "delay_difference",
    "has_departure_delay",
    "has_arrival_delay",
    "log_total_delay"
]

print("Created:")
print(delay_features)

print("\nStatistics:")
print(X_train_delay[delay_features].describe())

Created:
['total_delay', 'delay_difference', 'has_departure_delay', 'has_arrival_delay', 'log_total_delay']

Statistics:
         total_delay  delay_difference  has_departure_delay  \
count  559708.000000     559708.000000        559708.000000   
mean        2.307598         -0.044221             0.092405   
std        11.196878          3.340444             0.289598   
min         0.000000       -202.000000             0.000000   
25%         0.000000          0.000000             0.000000   
50%         0.000000          0.000000             0.000000   
75%         0.000000          0.000000             0.000000   
max       980.000000         91.000000             1.000000   

       has_arrival_delay  log_total_delay  
count      559708.000000    559708.000000  
mean            0.084165         0.305102  
std             0.277636         0.874313  
min             0.000000         0.000000  
25%             0.000000         0.000000  
50%             0.000000         0.000000  
75%

In [14]:
cat_cols_delay = [
    col for col in cat_cols
    if col in X_train_delay.columns
]

for col in cat_cols_delay:
    X_train_delay[col] = X_train_delay[col].astype("category")
    X_valid_delay[col] = X_valid_delay[col].astype("category")
    X_test_delay[col]  = X_test_delay[col].astype("category")

In [15]:
# ============================================================
# EXP 2 — LIGHTGBM WITH DELAY FEATURES
# ============================================================

lgb_delay = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,

    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=0.5,

    random_state=SEED,
    n_jobs=-1
)

lgb_delay.fit(
    X_train_delay,
    y_train,
    categorical_feature=cat_cols_delay,

    eval_set=[(X_valid_delay, y_valid)],

    callbacks=[
        lgb.early_stopping(100, verbose=False)
    ]
)

delay_valid_pred = lgb_delay.predict_proba(
    X_valid_delay
)[:, 1]

delay_auc = roc_auc_score(
    y_valid,
    delay_valid_pred
)

print("=" * 60)
print("EXP 2 — DELAY FEATURES")
print("=" * 60)

print(f"Baseline LGB AUC : {0.958134780:.9f}")
print(f"Delay LGB AUC    : {delay_auc:.9f}")
print(f"Delta             : {delay_auc - 0.958134780:+.9f}")
print(f"Best iteration    : {lgb_delay.best_iteration_}")

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


EXP 2 — DELAY FEATURES
Baseline LGB AUC : 0.958134780
Delay LGB AUC    : 0.957907015
Delta             : -0.000227765
Best iteration    : 558


## EXP 3 — High-Value Interaction Features

In [16]:
# ============================================================
# PHASE 5 — EXP 3A
# CLASS × TRAVEL TYPE
# ============================================================

def add_class_travel_interaction(df):
    df = df.copy()

    df["class_travel"] = (
        df["Class"].astype(str)
        + "_"
        + df["Type of Travel"].astype(str)
    )

    df["class_travel"] = df["class_travel"].astype("category")

    return df


X_train_int = add_class_travel_interaction(X_train)
X_valid_int = add_class_travel_interaction(X_valid)
X_test_int  = add_class_travel_interaction(X_test)

cat_cols_int = cat_cols + ["class_travel"]

for col in cat_cols_int:
    X_train_int[col] = X_train_int[col].astype("category")
    X_valid_int[col] = X_valid_int[col].astype("category")
    X_test_int[col]  = X_test_int[col].astype("category")

print(X_train_int["class_travel"].value_counts())

class_travel
Business_Business travel    270643
Eco_Personal Travel         148414
Eco_Business travel         113552
Eco Plus_Business travel     14039
Eco Plus_Personal Travel      9936
Business_Personal Travel      3124
Name: count, dtype: int64


In [17]:
# ============================================================
# EXP 3A — LIGHTGBM
# ============================================================

lgb_int = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,

    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=0.5,

    random_state=SEED,
    n_jobs=-1
)

lgb_int.fit(
    X_train_int,
    y_train,
    categorical_feature=cat_cols_int,

    eval_set=[(X_valid_int, y_valid)],

    callbacks=[
        lgb.early_stopping(100, verbose=False)
    ]
)

int_valid_pred = lgb_int.predict_proba(
    X_valid_int
)[:, 1]

int_auc = roc_auc_score(
    y_valid,
    int_valid_pred
)

print("=" * 60)
print("EXP 3A — CLASS × TRAVEL TYPE")
print("=" * 60)

print(f"Baseline LGB AUC : {0.958134780:.9f}")
print(f"Interaction AUC  : {int_auc:.9f}")
print(f"Delta             : {int_auc - 0.958134780:+.9f}")
print(f"Best iteration    : {lgb_int.best_iteration_}")

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


EXP 3A — CLASS × TRAVEL TYPE
Baseline LGB AUC : 0.958134780
Interaction AUC  : 0.958049267
Delta             : -0.000085513
Best iteration    : 477


## EXP 4 — Specialized LightGBM

In [19]:
# ============================================================
# CURRENT MODEL DIVERSITY CHECK
# ============================================================

print("Individual validation AUCs:")
print(f"LGB : {roc_auc_score(y_valid, lgb_valid):.9f}")
print(f"XGB : {roc_auc_score(y_valid, xgb_valid):.9f}")
print(f"CAT : {roc_auc_score(y_valid, cat_valid):.9f}")

pred_corr = pd.DataFrame({
    "LGB": lgb_valid,
    "XGB": xgb_valid,
    "CAT": cat_valid
}).corr()

print("\nPrediction correlations:")
print(pred_corr)

champion_valid = (
    0.57 * lgb_valid
    + 0.24 * xgb_valid
    + 0.19 * cat_valid
)

print("\nChampion:")
print(f"AUC: {roc_auc_score(y_valid, champion_valid):.9f}")

Individual validation AUCs:
LGB : 0.957874642
XGB : 0.957796183
CAT : 0.957697739

Prediction correlations:
          LGB       XGB       CAT
LGB  1.000000  0.997846  0.997972
XGB  0.997846  1.000000  0.997875
CAT  0.997972  0.997875  1.000000

Champion:
AUC: 0.958212191


In [20]:
lgb_valid_opt = np.load("artifacts/oof/lgb_optimized.npy")

## EXP 4B — Specialized Service Model

In [21]:
# ============================================================
# EXP 4 — SPECIALIZED SERVICE/PASSENGER MODEL
# ============================================================

specialized_cols = [
    "Age",
    "Flight Distance",
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class",

    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness"
]

X_train_spec = X_train[specialized_cols].copy()
X_valid_spec = X_valid[specialized_cols].copy()
X_test_spec  = X_test[specialized_cols].copy()

spec_cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

for col in spec_cat_cols:
    X_train_spec[col] = X_train_spec[col].astype("category")
    X_valid_spec[col] = X_valid_spec[col].astype("category")
    X_test_spec[col]  = X_test_spec[col].astype("category")

print("Specialized feature count:", len(specialized_cols))
print("Excluded:")
print(" - Departure Delay in Minutes")
print(" - Arrival Delay in Minutes")

Specialized feature count: 19
Excluded:
 - Departure Delay in Minutes
 - Arrival Delay in Minutes


In [22]:
# ============================================================
# EXP 4 — TRAIN SPECIALIZED LGB
# ============================================================

lgb_spec = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,

    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=0.5,

    random_state=SEED,
    n_jobs=-1
)

lgb_spec.fit(
    X_train_spec,
    y_train,
    categorical_feature=spec_cat_cols,

    eval_set=[(X_valid_spec, y_valid)],

    callbacks=[
        lgb.early_stopping(100, verbose=False)
    ]
)

spec_valid_pred = lgb_spec.predict_proba(
    X_valid_spec
)[:, 1]

spec_auc = roc_auc_score(
    y_valid,
    spec_valid_pred
)

print("=" * 60)
print("EXP 4 — SPECIALIZED SERVICE/PASSENGER MODEL")
print("=" * 60)

print(f"Specialized AUC : {spec_auc:.9f}")
print(f"Best iteration  : {lgb_spec.best_iteration_}")

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


EXP 4 — SPECIALIZED SERVICE/PASSENGER MODEL
Specialized AUC : 0.958063238
Best iteration  : 647


In [23]:
lgb_valid_opt = np.load(
    "artifacts/oof/lgb_optimized.npy"
)

print("=" * 60)
print("DIVERSITY ANALYSIS")
print("=" * 60)

print(f"Optimized LGB AUC : {roc_auc_score(y_valid, lgb_valid_opt):.9f}")
print(f"Specialized AUC   : {spec_auc:.9f}")

corr_spec_lgb = np.corrcoef(
    lgb_valid_opt,
    spec_valid_pred
)[0, 1]

print(f"\nLGB ↔ Specialized correlation: {corr_spec_lgb:.9f}")

DIVERSITY ANALYSIS
Optimized LGB AUC : 0.958134780
Specialized AUC   : 0.958063238

LGB ↔ Specialized correlation: 0.999290807


In [24]:
xgb_valid = np.load("artifacts/oof/baseline_xgb.npy")
cat_valid = np.load("artifacts/oof/baseline_cat.npy")

champion_valid = (
    0.57 * lgb_valid_opt
    + 0.24 * xgb_valid
    + 0.19 * cat_valid
)

champion_auc = roc_auc_score(
    y_valid,
    champion_valid
)

print(f"Champion AUC: {champion_auc:.9f}")

Champion AUC: 0.958338416


In [25]:
# ============================================================
# SPECIALIZED MODEL BLEND TEST
# ============================================================

best_auc = champion_auc
best_alpha = 0.0

for alpha in np.arange(0.00, 0.31, 0.01):

    # Take alpha away proportionally from existing champion
    new_pred = (
        (1 - alpha) * champion_valid
        + alpha * spec_valid_pred
    )

    auc = roc_auc_score(
        y_valid,
        new_pred
    )

    if auc > best_auc:
        best_auc = auc
        best_alpha = alpha

print("=" * 60)
print("SPECIALIZED MODEL BLEND")
print("=" * 60)

print(f"Champion AUC : {champion_auc:.9f}")
print(f"Best AUC     : {best_auc:.9f}")
print(f"Specialized weight: {best_alpha:.2f}")
print(f"Improvement  : {best_auc - champion_auc:+.9f}")

SPECIALIZED MODEL BLEND
Champion AUC : 0.958338416
Best AUC     : 0.958376553
Specialized weight: 0.25
Improvement  : +0.000038137


### EXP 4B — SPECIALIZED MODEL WEIGHT FINE SEARCH

In [26]:
# ============================================================
# EXP 4B — SPECIALIZED MODEL WEIGHT FINE SEARCH
# ============================================================

results = []

for alpha in np.arange(0.10, 0.351, 0.005):

    pred = (
        (1 - alpha) * champion_valid
        + alpha * spec_valid_pred
    )

    auc = roc_auc_score(y_valid, pred)

    results.append((alpha, auc))

results = sorted(
    results,
    key=lambda x: x[1],
    reverse=True
)

print("=" * 60)
print("EXP 4B — SPECIALIZED WEIGHT SEARCH")
print("=" * 60)

for alpha, auc in results[:15]:
    print(
        f"specialized={alpha:.3f} | "
        f"champion={1-alpha:.3f} | "
        f"AUC={auc:.9f}"
    )

EXP 4B — SPECIALIZED WEIGHT SEARCH
specialized=0.250 | champion=0.750 | AUC=0.958376553
specialized=0.245 | champion=0.755 | AUC=0.958376523
specialized=0.265 | champion=0.735 | AUC=0.958376473
specialized=0.255 | champion=0.745 | AUC=0.958376456
specialized=0.240 | champion=0.760 | AUC=0.958376421
specialized=0.275 | champion=0.725 | AUC=0.958376417
specialized=0.260 | champion=0.740 | AUC=0.958376392
specialized=0.280 | champion=0.720 | AUC=0.958376371
specialized=0.270 | champion=0.730 | AUC=0.958376368
specialized=0.285 | champion=0.715 | AUC=0.958376224
specialized=0.235 | champion=0.765 | AUC=0.958376218
specialized=0.230 | champion=0.770 | AUC=0.958376036
specialized=0.225 | champion=0.775 | AUC=0.958375928
specialized=0.290 | champion=0.710 | AUC=0.958375923
specialized=0.295 | champion=0.705 | AUC=0.958375823


## SAVE outputs

In [27]:
np.save(
    "artifacts/oof/lgb_specialized_service.npy",
    spec_valid_pred
)

In [30]:
spec_test_pred = lgb_spec.predict_proba(
    X_test_spec
)[:, 1]

np.save(
    "artifacts/test/lgb_specialized_service.npy",
    spec_test_pred
)

In [28]:
lgb_spec.booster_.save_model(
    "artifacts/lgb_specialized_service.txt"
)

In [29]:
import json

exp4_metadata = {
    "experiment": "EXP4_specialized_service_passenger",
    "validation_auc": float(spec_auc),
    "correlation_with_optimized_lgb": float(corr_spec_lgb),
    "blend_weight": 0.25,
    "champion_weight": 0.75,
    "blended_validation_auc": float(best_auc),
    "improvement": float(best_auc - champion_auc),
    "excluded_features": [
        "Departure Delay in Minutes",
        "Arrival Delay in Minutes"
    ],
    "best_iteration": int(lgb_spec.best_iteration_)
}

with open(
    "artifacts/exp4_specialized_metadata.json",
    "w"
) as f:
    json.dump(exp4_metadata, f, indent=2)

print("EXP 4 artifacts saved.")

EXP 4 artifacts saved.
